# 14a · 稳定性换成真正的能量（kcal/mol）

## 这一章在整个项目里的位置

```
00–13 ✅ 平台全部建成 → [第二阶段 F：把科学做深]
                          │
                          ├─ 14a 稳定性换成 kcal/mol        ← 你在这里
                          ├─ 14b 从测序读数到适应度
                          ├─ 14c 一个数拆成两个能量
                          ├─ 14d 少量数据 + 主动学习
                          ├─ 14e 位点地图
                          ├─ 14f 药理伴侣
                          └─ 14g 门面和讲法
```

## 背景：现在缺什么

09c 给全蛋白组 2.18 亿个突变各打了两个分：一个稳定性分、一个功能分。两个分都是 **0 到 1 之间的排名**——它只能说"这个突变比那个突变更糟"，说不出**糟多少**。

面试官和审稿人会问同一句话：**"这和 AlphaMissense 有什么区别？"** 现在的答案很弱，因为我们回答的是同一个问题（这个突变好不好），只是答案的算法不同。

这一章换一个问题。不再输出排名，而是输出一个**物理量**：

> **ΔΔG_fold**：这个突变让蛋白的折叠稳定性改变了多少 **kcal/mol**。

换成物理量有三个好处：

1. **可以和实验直接对账**。实验室测出来就是 kcal/mol，预测 1.2、实测 1.4，差 0.2——这句话排名是说不出来的。
2. **可以相加**。两个突变各自 1.0 和 0.8，合起来大约 1.8。排名没法相加，所以预测多突变必须有能量（14c 要用）。
3. **有现成的对手**。专门做稳定性预测的工具（ThermoMPNN、SPURS、RaSP）都在同一批数据、同一个切分上报过成绩，我们可以同题对照。

## 这一章做什么

| 步骤 | 内容 |
|---|---|
| 1 | 什么是折叠自由能，为什么它是"能量"而不是"分数" |
| 2 | 进货：Tsuboyama 2023 的大规模实验数据 + 别人发表的切分 |
| 3 | 把论文的过滤步骤一步步复现，四个数字必须对上 |
| 4 | **考卷会不会被弄脏**：这批数据和我们锁死的考试蛋白有没有同源 |
| 5 | 特征：ESM-2 + AlphaFold 结构（复用 07d、07e 的同一份代码） |
| 6 | **必须先赢的简单基线**："保守性 × 埋藏程度" |
| 7 | 训练：同样的网络，输出换成 kcal/mol |
| 8 | 考试，和已发表的数字同题对照 |

**不变的规矩**：测试蛋白锁死、考卷只开一次、负面结果照写。

---

## 准备

运行下面这一格：连上 Databricks（这一章只用它记账），载入代码，看看用哪个处理器训练。**不用看代码。**

In [ ]:
import json, os, subprocess, warnings, zipfile
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)
import boto3
import numpy as np
import pandas as pd
from databricks.sdk import WorkspaceClient
from qafi_platform import ingest, tracking
from qafi_core import stability, validation
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"   # 和 07–09 同一个实验
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
s3, lam = aws.client("s3"), aws.client("lambda")
RAW = json.loads(subprocess.run("terraform -chdir=infra/dev output -json bucket_names",
                                shell=True, capture_output=True, text=True, check=True).stdout)["raw"]
WORK = ROOT / "data/stability"        # 本机工作副本，不进 Git
WORK.mkdir(parents=True, exist_ok=True)
print("Databricks:", w.config.host)
print("这一章训练用的处理器:", stability.device())

### 结果怎么看
最后一行应该是 `mps`（Apple 芯片的 GPU）。这一章的训练只要十几秒，CPU 也完全够。

---

## 第 1 步 · 什么是折叠自由能

### 在干什么
弄清楚我们要预测的到底是什么物理量，以及它和前面章节的"分数"差在哪。

### 背景

一个蛋白在细胞里不是一动不动的，它在**折好**（folded）和**散开**（unfolded）两个状态之间来回。哪个状态多，由**自由能差** ΔG 决定：

```
散开的蛋白  ⇌  折好的蛋白
            ΔG
```

- **ΔG 越大，折好的越多**，蛋白越稳定。单位是 **kcal/mol**（千卡每摩尔），一个能量单位。
- 一个典型的小结构域，ΔG 大约 **3 到 5 kcal/mol**。
- **ΔΔG = 突变体的 ΔG − 野生型的 ΔG**：这个突变让稳定性变了多少。

直觉上的换算：在体温下，**ΔΔG 每差 1.4 kcal/mol，折好的比例差大约 10 倍**。所以一个 −3 kcal/mol 的突变基本上就是把蛋白拆了。

### 为什么这是"能量"而不是"分数"

| | 09c 的稳定性分 | 这一章的 ΔΔG |
|---|---|---|
| 取值 | 0–1 的排名 | kcal/mol，可正可负 |
| 比较对象 | 只能和同一个蛋白的其他突变比 | 可以和任何实验测的值比 |
| 能不能相加 | 不能 | 能（14c 要用） |
| 0 代表什么 | 这个蛋白里最差的那批 | 和野生型一样稳定 |

### 符号约定（很容易弄错，先说清楚）

- **Tsuboyama 原始数据**：ΔΔG **正 = 更稳定**。
- **ThermoMPNN**：代码里取了负号，所以它的**负 = 更稳定**。

我们**全程用 Tsuboyama 的约定**。相关系数的绝对值不受影响，但 kcal/mol 的误差受影响，所以报数字时必须写明用的哪一套。

### 面试怎么说
> "We predict ΔΔG of folding in kcal/mol rather than a 0-1 score, so the output is comparable with calorimetry, additive across mutations, and benchmarkable against the published stability predictors."

---

## 第 2 步 · 进货：数据从哪来

### 在干什么
把这一章要用的三份公开文件，按平台的规矩搬进 S3 raw。

### 背景：这批数据为什么特别

2023 年 Rocklin 实验室（Tsuboyama 等，《Nature》）用一种叫 **cDNA display proteolysis** 的方法，一次测了 **近百万个**蛋白变体的折叠稳定性。在这之前，整个领域几十年积累的实测 ΔΔG 加起来也就几千条。

它怎么测的，一句话：把蛋白暴露在**蛋白酶**里。折得好的蛋白不容易被剪断，折得差的很快被剪断。于是"需要多少蛋白酶才能剪断一半"这个浓度，就能换算成 ΔG。

规模：**479 个小结构域**（331 个天然的 + 148 个人工设计的），每个 40–72 个氨基酸，几乎每个位置的每种替换都测了。

### 用了什么技术，为什么选它

三份文件：

| 文件 | 来源 | 为什么要 |
|---|---|---|
| `Processed_K50_dG_datasets.zip`（1.0 GB） | Zenodo 记录 7992926 | 实测的 ΔG 和 ΔΔG 本身 |
| `AlphaFold_model_PDBs.zip`（14 MB） | 同上 | 每个结构域的结构，算埋藏程度要用 |
| `mega_splits.pkl` + `megascale_clusters.tsv` | ThermoMPNN 的 GitHub，版本钉死在一个 commit | **别人发表的训练/验证/测试切分** |

**为什么要用别人的切分，而不是自己切？**

这是这一章最重要的一个决定。ThermoMPNN 把 298 个结构域用 MMseqs2 按 **25% 序列一致性**聚类，整簇整簇地分到训练、验证、测试三边——这本身就是一个"按家族留出"的切分，正是我们要的。

更关键的是：**ThermoMPNN、SPURS、StaB-ddG 三篇论文都在这同一个切分的 28 个测试结构域上报过成绩**。沿用它，我们一行代码都不用跑，就有三个已发表的数字可以对照。自己切一个，这些对照全部作废。

### 一个要注意的安全问题：`.pkl` 文件

切分文件是一个 **pickle**。pickle 不是普通的数据文件，**它是一段程序**：Python 读它的时候会按文件里的指令去导入模块、调用函数。拿到一个来路不明的 pickle 直接 `pickle.load()`，等于运行一个陌生人给的脚本。

公司里的做法是两条：
1. **不收**。在数据进门的地方就拒绝这种文件类型。
2. **收，但用受限的读取器**。只允许一个白名单里的东西被导入，其他一律拒绝。

我们选第 2 条，因为这个文件是我们真正需要的官方产物。`qafi_core.stability` 里的读取器只放行三个 numpy 入口，别的一律报错——`tests/unit/test_stability.py` 里有一个测试专门拿一个"要求导入 pandas"的 pickle 去撞它，确认会被拒。验货员那边也相应加了 `.pkl` 的签名检查（文件头必须是 pickle 协议 4）。

### 执行
把三个数据源写进 `config/sources.toml`（已经写好了，这里只看一眼），然后让送货员 Lambda 去搬。

In [ ]:
sources = ingest.load_sources(ROOT / "config" / "sources.toml")
new = [s for s in sources if s["dataset"] in ("tsuboyama", "thermompnn")]
pd.DataFrame(new)[["dataset", "version", "key"]]

### 结果怎么看
四行，两个数据集。`key` 就是它们在 S3 里的位置：`datasets/<数据集>/<版本>/<文件名>`。版本写死了——Zenodo 的 `v2_230420`，和 ThermoMPNN 仓库的一个 40 位 commit 编号。

下面这一格真正去搬运（约 2–4 分钟，1 GB 那个文件最慢）。每个文件到位后，S3 事件会自动触发验货员检查类型、算校验和、写收货单。

In [ ]:
receipts = ingest.fetch_sources(lam, s3, RAW, new)

### 结果怎么看
每一行是一个文件的收货单：大小、校验和、`status: accepted`。1 GB 那个的大小应该正好是 1,013,678,473 字节。

下面把 raw 里的压缩包取一份到本机解开——raw 永远是正本，本机这份只是工作副本（和 08a 的 `data/models/` 一样，不进 Git）。

In [ ]:
KEEP = (".pdb", "Tsuboyama2023_Dataset2_Dataset3_20230416.csv")
for source in new:
    name = Path(source["key"]).name
    local = WORK / name
    if not local.exists():
        s3.download_file(RAW, source["key"], str(local))
    if name.endswith(".zip"):
        with zipfile.ZipFile(local) as z:
            wanted = [n for n in z.namelist() if not n.startswith("__MACOSX") and n.endswith(KEEP)]
            for member in wanted:
                target = WORK / Path(member).name
                if not target.exists():
                    target.write_bytes(z.read(member))
        print(f"{name}: 解出 {len(wanted)} 个文件")
    else:
        print(f"{name}: 已就位")
print("工作副本:", WORK)

---

## 第 3 步 · 复现论文的过滤，四个数字必须对上

### 在干什么
按 ThermoMPNN 论文写的步骤，把近百万行原始数据筛成能训练的那一份，并且**每一步的行数都和论文对上**。

### 背景：为什么要一个数字一个数字地对

你用的是别人的数据和别人的切分，那就必须证明**你手里的那份和他们手里的那份是同一份**。只要有一步筛法不同，后面所有"同题对照"就都是假的——你赢了可能只是因为你的测试集更容易。

公司里这叫**对账**（reconciliation）。做数据的岗位，这是基本功：每做一次筛选，都要有一个能和外部核对的数字。

论文写的三步：

| 步骤 | 去掉什么 | 剩下 |
|---|---|---|
| 0 | 原始文件（数据集 2 + 3） | 776,298 |
| 1 | `ddG_ML` 是 `-` 的（拟合不可靠，给不出能量） | 607,839 |
| 2 | 插入、缺失、多突变 | 391,090 |
| 3 | 只留切分里那 298 个结构域（等于排掉作者另外测的"扰动过的野生型"背景） | 272,712 |

### 执行

In [ ]:
splits = stability.load_splits(WORK / "mega_splits.pkl")
muts = stability.megascale(WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv", splits)
print({k: len(v) for k, v in splits.items()}, "个结构域")
summary = muts.groupby("split").agg(结构域=("domain", "nunique"), 突变=("ddg", "size"))
print(summary.to_string())
print("\n论文公布的数字:", stability.PUBLISHED_COUNTS["domains"],
      "| 考试集", stability.PUBLISHED_COUNTS["test_rows"], "行（含 140 行野生型）=",
      stability.PUBLISHED_COUNTS["test_mutations"], "个突变")
muts.head(3)

### 结果怎么看

三边的结构域数应该正好是 **239 / 31 / 28**，考试集 **28,172 个突变**。

论文写的是"test = 28,312"，我们算出 28,172，差 140。查下来是：**论文那个数字把 140 行野生型自己也算进去了**（每个结构域测了好几次野生型作为对照）。模型真正要预测的是 28,172 个突变。这种差一点点的地方，查清楚再往下走，不要糊过去——`PUBLISHED_COUNTS` 里两个数都记着。

`megascale()` 还顺手做了一个检查：每一行写的野生型氨基酸，必须和序列在那个位置上的氨基酸一致。对不上就报错。编号错位是这类数据最常见、也最安静的坑（07c 做 DMS 编号换算时踩过）。

### 面试怎么说
> "I reproduced the published filtering chain row-count by row-count before using the data, and ran down a 140-row discrepancy in the paper's test-set size: it counts wild-type control rows."

---

## 第 4 步 · 考卷会不会被弄脏

### 在干什么
检查这批新数据里，有没有和**我们自己锁死的 108 个考试蛋白**同源的结构域。有的话必须从训练里拿掉。

### 背景：两套考卷撞在一起

这一章引入了一个**新的**数据集，它自带一套训练/测试切分。但我们从 07a 起就锁死了**我们自己的**一套考试蛋白，后面 14c、14e 还要用。

危险在这里：

```
Tsuboyama 的训练结构域  ──同源──▶  我们锁死的考试蛋白
         ↑                              ↑
   14a 在这上面训练              14c 要在这上面考试
```

如果 14a 的模型见过某个结构域，而那个结构域和我们的考试蛋白是同一个家族，那 14c 用这个模型时，考卷就已经泄漏了。

**这不是理论上的担心。** 这批数据里有大量常见小折叠：SH3 结构域、同源框、PDZ、villin 头部、染色质域——而我们的考试蛋白里正好有不少人类蛋白带着这些结构域。

### 用了什么技术

不用重新发明。`qafi_core.validation.related()` 就是 07a/07c 判断"两个蛋白算不算亲戚"的那个函数：局部比对，三个条件同时满足才算亲戚（E 值 < 1e-5、一致性 ≥ 30%、覆盖短的那条的一半以上）。这是序列聚类工具（MMseqs2、CD-HIT）的标准口径。

**同一把尺子量到底**，是这件事唯一能说清楚的办法。

### 执行
298 个结构域 × 108 个考试蛋白，逐对比对（几秒）。

In [ ]:
split_table = pd.read_csv(ROOT / "dbt/seeds/protein_split.csv")
sequences = pd.read_parquet(ROOT / "data/models/proteome_sequences.parquet").set_index("uniprot_id").sequence
locked = {u: sequences[u] for u in split_table.loc[split_table.split == "test", "uniprot_id"] if u in sequences.index}
domains = muts.groupby(["domain", "split"]).seq.first().reset_index()
hits = pd.DataFrame(
    [{"domain": d.domain, "domain_split": d.split, "test_protein": u}
     for d in domains.itertuples() for u, s in locked.items() if validation.related(d.seq, s)]
)
related = set(hits.loc[hits.domain_split.isin(["train", "val"]), "domain"])
print(f"和我们考试蛋白同源的结构域: {hits.domain.nunique()} / {len(domains)}")
print(hits.groupby("domain_split").domain.nunique().to_string())
print(f"被牵连的考试蛋白: {hits.test_protein.nunique()} / {len(locked)}")
hits.groupby("test_protein").domain.count().sort_values(ascending=False).head(8)

### 结果怎么看

大约 **56 个结构域**和我们的考试蛋白同源，其中 **52 个在 Tsuboyama 的训练侧**，牵连到 **29 个**我们的考试蛋白。数量相当可观，说明这个检查不是走过场。

### 怎么处理：训练两个模型

拿掉这 52 个结构域会损失 20% 的训练数据。但如果为了成绩好看而不拿掉，我们自己的考卷就废了。所以这一章**两个都训**：

| 模型 | 训练数据 | 用途 |
|---|---|---|
| **可比版** | 全部 270 个结构域（ThermoMPNN 的训练 + 验证） | 只为了和已发表数字公平对照；**绝不**用于后面的章节 |
| **干净版** | 218 个（去掉同源的 52 个） | 14c 以后用这一个 |

两个都在同样的 28 个考试结构域上报成绩。两者的差距，就是"保持考卷干净"的真实代价——这个数字本身就值得写出来。

### 面试怎么说
> "A second dataset brings its own split, which can silently contaminate the project's locked test set. I measured the overlap with the same homology rule used everywhere else, trained one model for the published comparison and a decontaminated one for everything downstream, and reported the gap between them."

---

## 第 5 步 · 特征：复用已有的两套

### 在干什么
给这 298 个结构域算特征。**不写新代码**，用 07d（结构）和 07e（语言模型）里已经在跑的那两份。

### 背景

每个突变的输入由三部分组成：

| 来源 | 内容 | 代码在哪 |
|---|---|---|
| **ESM-2 650M** | 这个残基的 1,280 维向量；20 种氨基酸各自的对数概率 | `databricks/src/lakehouse/language_model.py` |
| **AlphaFold 结构** | 埋藏程度 rsa、pLDDT、8 Å / 12 Å 接触数、上下半球暴露度 | `databricks/src/lakehouse/structure.py` |
| **算出来的** | 替换的对数概率差、野生型的对数概率、位置的熵 | `qafi_core.stability.add_features` |

**一个重点**：结构用的是 **Tsuboyama 自己提供的 AlphaFold 模型**，不是我们 07d 下载的人类蛋白组结构。因为这批结构域里有 148 个是**人工设计的蛋白**，世界上本来就没有，必须用作者给的。298 个结构域全都有对应的结构文件，而且文件里的序列和数据表里的序列逐个核对过，一个不差。

### 执行
结构特征约 1 分钟，ESM-2 约半分钟（序列很短）。

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "databricks" / "src"))
from lakehouse import language_model as lm, structure as struct

domains["stem"] = domains.domain.str.split(".pdb").str[0].str.replace("|", ":", regex=False)
if not (WORK / "structure.parquet").exists():
    frames = []
    for d in domains.itertuples():
        f = struct.fragment_features((WORK / f"{d.stem}.pdb").read_text())
        frames.append(f.drop(columns=["neighbours", "distances", "ca_x", "ca_y", "ca_z"]).assign(domain=d.domain))
    pd.concat(frames, ignore_index=True).to_parquet(WORK / "structure.parquet")
if not (WORK / "logprobs.npy").exists():
    model, tokenizer = lm.load()                       # 和 07e 同一个钉死的版本
    read = lm.read_proteins(list(domains.seq), model, tokenizer, embeddings=True)
    np.save(WORK / "logprobs.npy", np.concatenate([r[0] for r in read]))
    np.save(WORK / "vectors.npy", np.concatenate([r[1] for r in read]).astype(np.float16))
    pd.DataFrame({"domain": np.repeat(domains.domain.values, domains.seq.str.len().values),
                  "position": np.concatenate([np.arange(1, len(s) + 1) for s in domains.seq])}
                 ).to_parquet(WORK / "residue_index.parquet")
structure = pd.read_parquet(WORK / "structure.parquet")
index = pd.read_parquet(WORK / "residue_index.parquet")
logprobs, vectors = np.load(WORK / "logprobs.npy"), np.load(WORK / "vectors.npy")
df = stability.add_features(muts, structure, index, logprobs)
print(f"残基 {len(index):,} | 向量 {vectors.shape} | 带特征的突变 {len(df):,}")
df[["domain", "mut_type", "ddg"] + stability.FEATURES].head(3)

### 结果怎么看
15,193 个残基、1,280 维向量、271,231 个突变，没有缺失值。`add_features` 还检查了结构文件里的氨基酸和突变表里的野生型一致——不一致就报错，不会悄悄算出一堆错特征。

---

## 第 6 步 · 必须先赢的那个简单基线

### 在干什么
先算一个**不用训练、一个参数都没有**的基线，作为这一章的及格线。

### 背景：一篇让人清醒的论文

2025 年有一篇论文，标题直接就是结论：**《残基保守性和溶剂可及性（几乎）就是你需要的全部》**（Tsishyn 等，*Bioinformatics*）。它的做法简单到离谱：

```
分数 = (1 − 相对溶剂可及度) × 保守性的对数几率
```

翻译成人话：**一个氨基酸埋得越深、在进化上越保守，换掉它越要命。** 两个数相乘，没有任何需要训练的参数。

在 ProteinGym 上，这个式子打赢了 27 个无监督模型里的 26 个。

**为什么要先跑它**：如果一个几千万参数的网络赢不过这两个数相乘，那它学到的东西就是可疑的。很多论文不报这个基线，报了的话排行榜会难看很多。这是这一章最有说服力的一格。

### 我们的版本和原文的区别（要说明白）

原文的"保守性"来自**多序列比对**的对数几率比。我们这里用 **ESM-2 的对数概率差**代替——因为 07e 已经把它算好了，不用再为 298 个结构域建比对。这是一个**替代品**，不是原方法，讲的时候要说清楚。

### 执行

In [ ]:
test = df[df.split == "test"].copy()
rows = [{"方法": "ESM-2 对数概率差（只有保守性）", **stability.score(test, "esm_delta")},
        {"方法": "埋藏程度（只有结构）", **stability.score(test, "buried")},
        {"方法": "埋藏 × 保守性（必须赢的基线）", **stability.score(test, "rsalor")}]
pd.DataFrame(rows)[["方法", "spearman", "pearson", "median_spearman"]].round(3)

### 结果怎么看

| 方法 | 总体 ρ | 每结构域中位 ρ |
|---|---|---|
| 只有保守性 | 0.277 | 0.456 |
| 只有埋藏程度 | −0.461 | −0.514 |
| **两个相乘** | **0.442** | **0.582** |

两个弱信号相乘，比各自都强得多。注意埋藏程度是**负相关**（越埋越容易被破坏），这符合物理直觉。

这里**没有报 RMSE**，因为这三个基线输出的不是 kcal/mol，拿它们算"误差多少千卡"没有意义。只有真正预测能量的模型才配报 RMSE——这一点很多排行榜都混着写。

**这一章的及格线定了：总体 Spearman 要明显超过 0.442。**

---

## 第 7 步 · 训练：同样的网络，输出换成能量

### 在干什么
训练两个模型（可比版和干净版），输出直接就是 kcal/mol。

### 背景：和 08a 的网络差在哪

网络结构几乎原封不动搬 08a 的：ESM 向量先做 LayerNorm，拼上几个特征，过 512 → 128，最后一层给 20 个输出（每种氨基酸一个），只取突变成的那一种。

**只有一处不同，但很关键**：

| | 08a | 14a |
|---|---|---|
| 目标 | 每组实验各自标准化后的排名 | **kcal/mol 本身** |
| 损失 | 标准化尺度上的均方误差 | **物理单位上的均方误差** |
| 输出的含义 | 一个相对名次 | **一个能量** |

因为目标本身就有统一的物理单位，**不需要再做"每组实验各自标准化"**那一套了。08a 之所以要标准化，是因为几百组实验的尺度互不相同；这里所有数据来自同一种测量方法，单位天然一致。这正是 14c 要推广的思路。

### 训练多少轮：用验证集定，不是拍脑袋

切分里专门留了 31 个验证结构域。先只用训练侧的 239 个训练，每轮在验证的 31 个上看成绩：

| 轮数 | 验证 ρ | 验证中位 ρ | 验证 RMSE |
|---|---|---|---|
| 1 | 0.702 | 0.726 | 0.625 |
| 2 | 0.753 | 0.777 | 0.575 |
| **3** | **0.764** | **0.797** | **0.571** |
| 4 | 0.763 | 0.795 | 0.576 |
| 8 | 0.759 | 0.802 | 0.589 |
| 24 | 0.762 | 0.791 | 0.591 |

第 3–4 轮见顶，之后训练损失还在降但验证不再变好——**过拟合**，和 08a 的学习曲线是一回事。所以 `SETTINGS["epochs"] = 4`。轮数定下来之后，再把验证集并进训练（这是标准做法：选超参用验证集，定好了就把数据用满）。

另外跑三个随机种子取平均（08a 验证过这招能稳稳提一点，而且只多花几秒）。

### 执行
两个模型各三个种子，一共约 25 秒。

In [ ]:
train_all = df[df.split.isin(["train", "val"])]
train_clean = train_all[~train_all.domain.isin(related)]
models = {}
for name, data in [("comparable", train_all), ("clean", train_clean)]:
    models[name] = stability.fit_ensemble(data, vectors)
    test[name] = stability.predict_ensemble(models[name], test, vectors)
    print(f"{name:11s} 训练于 {data.domain.nunique():3d} 个结构域 / {len(data):7,} 个突变")

---

## 第 8 步 · 考试：和已发表的数字同题对照

### 在干什么
在 28 个从没见过的结构域上出成绩，和三个已发表的工具比。

### 对手是谁

| 工具 | 出处 | 怎么做的 |
|---|---|---|
| **ThermoMPNN** | PNAS 2024 | ProteinMPNN（一个专门读蛋白结构的网络）+ 迁移学习 |
| **SPURS** | Nat Commun 2025 | 冻结的 ESM2-650M + ProteinMPNN，用 cross-attention 重新接线 |
| 埋藏 × 保守性 | Bioinformatics 2025 | 两个数相乘，零参数 |

它们的数字是**从论文里抄的**，不是我们跑的——因为用的是同一个切分、同一批 28 个考试结构域，所以可以直接比。这就是第 2 步坚持用别人切分的回报。

### 执行

In [ ]:
table = [{"模型": "埋藏 × 保守性（零参数基线）", **stability.score(test, "rsalor")},
         {"模型": "我们的（可比版，270 个结构域）", **stability.score(test, "comparable")},
         {"模型": "我们的（干净版，218 个结构域）", **stability.score(test, "clean")}]
out = pd.DataFrame(table)[["模型", "spearman", "pearson", "median_spearman", "rmse"]]
out.loc[out.模型.str.startswith("埋藏"), "rmse"] = np.nan      # 它不输出 kcal/mol，误差无意义
published = pd.DataFrame(stability.PUBLISHED_TEST_SCORES).T.reset_index(names="模型")
published["模型"] += "（已发表）"
print(f"考试集: {len(test):,} 个突变 / {test.domain.nunique()} 个结构域 · 符号用 Tsuboyama 约定\n")
pd.concat([out, published], ignore_index=True)[
    ["模型", "spearman", "pearson", "median_spearman", "rmse"]].round(3)

### 结果怎么看

| 模型 | 总体 ρ | 总体 r | 每结构域中位 ρ | RMSE (kcal/mol) |
|---|---|---|---|---|
| 埋藏 × 保守性（零参数） | 0.442 | 0.515 | 0.582 | — |
| **我们（可比版）** | **0.691** | **0.723** | **0.764** | **0.736** |
| **我们（干净版）** | **0.681** | **0.711** | **0.759** | **0.744** |
| ThermoMPNN（已发表） | 0.725 | 0.754 | 0.77 | 0.708 |
| SPURS（已发表） | 0.74 | 0.78 | 0.83 | — |

三件事：

**1. 简单基线被远远甩开**（0.442 → 0.691）。及格线过了，而且过得很明显。网络确实学到了两个数相乘学不到的东西。

**2. 和 ThermoMPNN 基本打平，略低。** 总体 Spearman 0.691 对 0.725，差 0.034；但**每结构域中位数 0.764 对 0.77，几乎一样**。后面这个指标更接近实际用法——你拿到一个蛋白，关心的是"在这个蛋白内部排得准不准"。

  诚实地说：我们**没有**赢 SPURS（0.74 / 0.83）。它是专门为这个任务设计的架构，而我们用的是 08a 那个通用网络，一行结构编码器都没加。差距合理。

**3. 保持考卷干净只花了 0.010。** 可比版 0.691，干净版 0.681。为了不污染我们自己的考试集，丢掉 20% 的训练数据，代价只有 0.01 个 Spearman。**这笔买卖非常划算**，后面所有章节都用干净版。

### 这一章的诚实结论

> 用一个通用网络（冻结的 ESM-2 + 几个结构特征 + 三层全连接），在和专用工具完全相同的考题上，拿到了和 ThermoMPNN 相当的每蛋白成绩，落后 SPURS。真正的收获不是名次，而是**输出从此是 kcal/mol**：可以和实验对账，可以相加，14c 才能把它拆成两个能量。

### 面试怎么说
> "On the published mega-scale split, our generic ESM-2 head reaches a median per-domain Spearman of 0.76 against ThermoMPNN's published 0.77 and SPURS' 0.83, and an RMSE of 0.74 kcal/mol against ThermoMPNN's 0.71 — while beating the zero-parameter conservation-times-burial baseline by a wide margin, which is the comparison most stability papers skip."

---

## 第 9 步 · 记账

### 在干什么
把这次的设置、数字和每个结构域的成绩记进 MLflow，和 07–09 在同一个实验里。

### 背景
"在哪训练不要紧，账要记在一个地方"——这是 08a 定下的规矩。训练跑在你的 Mac 上，记录进 Databricks 的 MLflow。这样三个月后有人问"那个 0.76 是怎么来的"，能查到代码版本、每个参数、每个结构域的成绩。

这一章用 `tracking.experiment_run`：07–09 用的 `cv_run` 要求"每组标注数据一个分数"，而这一章的单位是另一个数据集的结构域，对不上，所以单开一个记法，但进同一个实验、带同样的代码版本标签。

In [ ]:
for name in ["comparable", "clean"]:
    data = train_all if name == "comparable" else train_clean
    tracking.experiment_run(
        EXPERIMENT, f"stability_kcal_{name}",
        params={**stability.SETTINGS, "seeds": len(stability.SEEDS), "features": len(stability.FEATURES),
                "train_domains": int(data.domain.nunique()), "train_mutations": len(data),
                "excluded_as_relatives": 0 if name == "comparable" else len(related),
                "split": "ThermoMPNN mega_splits.pkl", "sign": "Tsuboyama (positive = stabilising)"},
        metrics={k: float(v) for k, v in stability.score(test, name).items()},
        tables={"per_domain": stability.per_domain(test, name)},
        chapter="14a")
print("已记录两次运行")
runs = tracking.latest_runs(EXPERIMENT)
runs[runs["tags.chapter"] == "14a"][["metrics.median_spearman", "metrics.rmse", "params.train_domains", "tags.code_version"]]

---

## 这一章做完了什么

- 把 Tsuboyama 2023 的近百万条折叠稳定性按平台规矩进了 S3 raw，版本钉死；`.pkl` 用受限读取器读，验货员加了对应的类型检查
- **逐步复现了已发表的过滤链**（776,298 → 607,839 → 391,090 → 272,712），并查清了论文测试集数字差 140 的原因
- **查出了跨数据集的考卷污染**：56 个结构域和我们锁死的考试蛋白同源，训练两个模型把这件事量化（代价只有 0.010 Spearman）
- 稳定性预测从"排名"变成 **kcal/mol**：每结构域中位 Spearman 0.76、RMSE 0.74 kcal/mol，和 ThermoMPNN 相当，落后 SPURS，远超零参数基线

## 下一步

**14b · 从测序读数到适应度**：这一章用的是别人算好的能量。下一章回到更上游——拿一个深度突变扫描实验的**原始测序 FASTQ**，用 Nextflow 包住 DiMSum 跑成适应度分数，和作者公布的对上。那是生信岗位面试真正会考的东西。

之后 **14c** 才是重头：把一个数拆成**两个**能量。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14a stability in kcal/mol on the published mega-scale split' && git log --oneline -1")